# Stellar Classification and Supervised Learning
This notebook demonstrates how to build, train, and evaluate multiple models (both traditional ML and PyTorch Neural Networks) using our custom `stellar_classification` package.

In [1]:
import sys
import os
import gc
import pandas as pd
import joblib
import matplotlib.pyplot as plt

#sistemare il warning di LightGBM fa creare warning a tutti gli altri modelli addestrati senza
#feature name, quindi meglio ingorarli
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Add the package root to sys.path so we can import stellar_classification
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'stellar_classification')))

import stellar_classification as sc

# Enable garbage collection
gc.enable()
gc.collect()

20

In [ ]:
# Load dataset
data_path = '../stellar_classification/stellar_classification/data/star_classification.csv'
star = pd.read_csv(data_path)

print("First few rows:")
display(star.head())

print("\nData Info:")
star.info()

print("\nNull Values:")
print(star.isnull().sum())

print("\nClass Distribution:")
print(star["class"].value_counts(normalize=True) * 100)

# -------- Color indexes ----------
star["u_g"] = star["u"] - star["g"]
star["g_r"] = star["g"] - star["r"]
star["r_i"] = star["r"] - star["i"]
star["i_z"] = star["i"] - star["z"]

star.info()


In [ ]:
sc.plot_class_distribution(star['class'], title="Stellar Class Distribution")

In [4]:
# Preprocessing: Apply outlier removal, splits, standardization, and SMOTE
X_train, X_val, X_test, y_train, y_val, y_test, label_encoder, scaler, feature_names = sc.prepare_splits(
    star, 
    target_col='class', 
    test_size=0.2, 
    val_ratio=0.25, 
    random_state=42, 
    apply_outlier_removal=True
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")


Outliers removed: 17,661 rows  (82,339 remain)
X_train shape: (96288, 14)
X_val shape: (16468, 14)
X_test shape: (16468, 14)


In [5]:
# Convert numpy arrays to PyTorch DataLoaders
train_loader, val_loader, test_loader = sc.to_dataloaders(
    X_train, y_train, 
    X_val, y_val, 
    X_test, y_test, 
    batch_size=64
)

print("DataLoaders created.")


DataLoaders created.


In [6]:
# Train and evaluate traditional ML models
models = sc.train_traditional(X_train, y_train, X_val, y_val)


Linear SVC trained.
  [Training] Acc=93.76%  P=0.94  R=0.94  F1=0.94
  [Validation] Acc=92.03%  P=0.86  R=0.94  F1=0.89
Decision Tree trained.
  [Training] Acc=100.00%  P=1.00  R=1.00  F1=1.00
  [Validation] Acc=95.70%  P=0.91  R=0.95  F1=0.93


KeyboardInterrupt: 

In [ ]:
# Train Voting Classifier utilizing the trained models and save it in trained_models directory
voting_clf = sc.train_voting(X_train, y_train, X_val, y_val, models=models)



In [20]:
# Train PyTorch Neural Network
import torch

input_size = X_train.shape[1]
num_classes = len(label_encoder.classes_)

nn_model = sc.train_neural(
    train_loader=train_loader,
    val_loader=val_loader,
    input_size=input_size,
    num_classes=num_classes,
    num_epochs=10,
    lr=0.001
)

#save the model in trained_models directory
torch.save(nn_model.state_dict(), 'trained_models/nn_model_redshift.pth')
print('model trained and saved')

Epoch  1/10  loss=0.2545  val_acc=93.71%
Epoch  2/10  loss=0.1619  val_acc=94.13%
Epoch  3/10  loss=0.1458  val_acc=94.19%
Epoch  4/10  loss=0.1376  val_acc=95.20%
Epoch  5/10  loss=0.1322  val_acc=94.97%
Epoch  6/10  loss=0.1280  val_acc=94.15%
Epoch  7/10  loss=0.1252  val_acc=95.12%
Epoch  8/10  loss=0.1229  val_acc=94.91%
Epoch  9/10  loss=0.1210  val_acc=95.81%
Epoch 10/10  loss=0.1191  val_acc=95.07%
model trained and saved


In [ ]:
# Base inference on test set
import torch 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

y_test_pred_voting = voting_clf.predict(X_test)
voting_metrics = sc.evaluate_test_set(y_test, y_test_pred_voting, "Voting Classifier")
sc.print_metrics(voting_metrics)

nn_metrics = sc.evaluate_neural(test_loader, nn_model, device, "Neural Network")
sc.print_metrics(nn_metrics)



In [ ]:
# Confusion Matrices
class_names = list(label_encoder.classes_)

sc.plot_confusion_matrix(voting_metrics['confusion_matrix'], class_names=class_names, title='Confusion Matrix - Voting Classifier')
sc.plot_confusion_matrix(nn_metrics['confusion_matrix'], class_names=class_names, title='Confusion Matrix - Neural Network')


In [ ]:
# Feature Importance (using permutation importance on the Voting Classifier)
#n_jobs decide quanti core usare, parti da 1 e monitora la ram, aumentalo un po alla volta (con 16 GB non oltre n_jobs=3)
#n_jobs è un parametro di permutation_importance di scikit-learn, leggete 
#la doc, non mettete n_jobs=-1 !!!!!!
imp = sc.compute_permutation_importance(voting_clf, X_test, y_test, feature_names=feature_names, n_jobs=6)
sc.plot_permutation_importance(imp, top_n=10)

print("Feature Importance Scores:")
for feat, val in imp.items():
    print(f"{feat}: {val:.4f}")
